This notebook is created to demonstrate how to use the MDM. It may or may not be updated in the future. The current use of the MDM is as a huge vector generator, but in the future, it should be recreated as a combiniation of basis functions and the likelihood should be changed to the KL divergence between the modeled and real transfer function

Thanks for spending the time going through to decode :)

In [ ]:
import PREDICTION as por #Importing PREDICTION_onePDF module from the package
from PREDICTION import * #Importing importing all packages from PREDICTION_onePDF module
#The functions plot_function2, back_x and back_y must be imported separately
from PREDICTION import plot_function2, back_x, back_y, find_LC_transform 
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import TensorDataset, DataLoader, Subset
import torch.optim as optim
from torch.distributions import Normal
import seaborn as sns
from MDM import train_and_evaluate

In [ ]:
#Model hyperparameters
#Match this to the previous model's hyperparameters
encoding_size = 256 #Encoder MLP layer size
latent_size = 256 #Latent dimension size
no_latent_samples = 10 #The number of samples of the latent space to take

attention_type = 'scaledot' #Can also use multihead, but scaledot works better
cross_attention = True #Whether to include cross-attention in the deterministic path
self_attention = True #Whether to include self-attention in both paths

lstm_layers = 0 #The number of LSTM layers to use for pre-encoding
lstm_size = 32 #The size of the LSTM layer

replace_lstm_with_gru = False # Whether to use a GRU instead of an LSTM
bidirectional = False #Whether to use bidirectional LSTM/GRU layers
lstm_agg = False #Whether to aggregate the latent space representations via an LSTM instead of mean pooling
activation = 'relu' #Can also make it 'leaky' for LeakyReLu but ReLu seems to work better
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #defining the device for testing, it can be CPU or CUDA

In [ ]:
all_full_rep_bands = []
all_z_bands = []
all_transfers_funcs = []
num_target_smooth = 730
for band in 'ugriz':
    print(f'Band:{band}')
    full_folder = './'
    suffix = f'LCs_{band}' #Whatever suffix you gave to the output files
    #The paths for the transfer function and parameters if they exist
    tf_and_param = True #Set false if using real data without transfer function and parameter
    if tf_and_param:
        TF_PATH = f'Transfer_Functions/{band}/'
        #param_df_path = 'Parameters.csv' #If there is no parameters dataframe, it should be None
        param_df_path = 'Parameters.csv'
        param_df = pd.read_csv(param_df_path)
        scaled_param_df = pd.read_csv(param_df_path)
        param_columns=['Log_Mass','Inclination','Log_Tau','z','Eddington_Ratio','SFinf'] #Change to the names of your columns
        param_length = len(param_columns)
        TF_SAVE_PATH_TRAIN = f'{full_folder}/output_{suffix}/predictions/Transfer_Functions/train/'
        TF_SAVE_PATH_TEST = f'{full_folder}/output_{suffix}/predictions/Transfer_Functions/test/'
        TF_SAVE_PATH_VAL = f'{full_folder}/output_{suffix}/predictions/Transfer_Functions/val/'
        PARAM_SAVE_PATH_TRAIN = f'{full_folder}/output_{suffix}/predictions/Parameter_Predictions/train/'
        PARAM_SAVE_PATH_VAL = f'{full_folder}/output_{suffix}/predictions/Parameter_Predictions/val/'
        PARAM_SAVE_PATH_TEST = f'{full_folder}/output_{suffix}/predictions/Parameter_Predictions/test/'
        beta_param = 0 #Change to the value you trained
        beta_tf = 0 #Change to the value you trained
        transfer_function_length = 1000 #Change to the appropriate tf_length
    #Creating the predictions folder
    os.makedirs(f'{full_folder}/output_{suffix}/predictions',exist_ok=True)
    os.makedirs(f'{full_folder}/output_{suffix}/predictions/Transfer_Functions/test',exist_ok=True)
    os.makedirs(f'{full_folder}/output_{suffix}/predictions/Transfer_Functions/train',exist_ok=True)
    os.makedirs(f'{full_folder}/output_{suffix}/predictions/Transfer_Functions/val',exist_ok=True)
    os.makedirs(f'{full_folder}/output_{suffix}/predictions/Parameter_Predictions/test',exist_ok=True)
    os.makedirs(f'{full_folder}/output_{suffix}/predictions/Parameter_Predictions/train',exist_ok=True)
    os.makedirs(f'{full_folder}/output_{suffix}/predictions/Parameter_Predictions/val',exist_ok=True)
    por.create_prediction_folders(f'{full_folder}/output_{suffix}/predictions')

    #deleting the padded values from test set
    folder_path = f'{full_folder}/dataset_{suffix}/test/'
    por.remove_padded_values_and_filter(folder_path)
    
    folder_path = f'{full_folder}/dataset_{suffix}/train/' 
    por.remove_padded_values_and_filter(folder_path)
    
    folder_path = f'{full_folder}/dataset_{suffix}/val/'  
    por.remove_padded_values_and_filter(folder_path)

    MODEL_PATH = f"{full_folder}/output_{suffix}/model_{suffix}.pth" #path to saved model
    #Path to the data
    DATA_PATH_TRAIN = f"{full_folder}/dataset_{suffix}/train"
    DATA_PATH_VAL = f"{full_folder}/dataset_{suffix}/val"
    DATA_PATH_TEST= f"{full_folder}/dataset_{suffix}/test"
    OUTPUT_PATH = f'{full_folder}/output_{suffix}/predictions/' #path where to save the plots and data    

    #clearing outputh dir if you want. Uncomment the next line
    clear_output_dir=por.prepare_output_dir(OUTPUT_PATH)
    
    #Load the saved model
    model = por.load_trained_model(MODEL_PATH, device, encoding_size,latent_size,latent_mlp_size=encoding_size,attention = cross_attention,self_attention=self_attention,no_latent_space_sample=no_latent_samples,lstm_layers = lstm_layers,lstm_agg = lstm_agg,lstm_size=lstm_size,transfer_function_length=0,parameters_length = 0,classes = 0,replace_lstm_with_gru=replace_lstm_with_gru
                                    ,activation=activation,bidirectional=bidirectional)
    #loading criterion and metrics
    criterion, mseMetric=por.get_criteria()

    #loading the coeffitients to transform back from [-2,2] to the real values
    tr=por.load_trcoeff(f'{full_folder}/TR_Coeffs/trcoeff_{suffix}.pickle')

    #Load train data
    trainLoader=por.load_train_data(DATA_PATH_TRAIN,num_target_smooth=num_target_smooth,tf_dir = TF_PATH,param_df=param_df,param_columns=param_columns,class_labels_df = None)
    #Load test data
    testLoader=por.load_test_data(DATA_PATH_TEST,num_target_smooth=num_target_smooth,tf_dir = TF_PATH,param_df=param_df,param_columns=param_columns,class_labels_df = None)
    #Load val data
    valLoader=por.load_val_data(DATA_PATH_VAL,num_target_smooth=num_target_smooth,tf_dir = TF_PATH,param_df=param_df,param_columns=param_columns,class_labels_df = None)

    #Defining empty lists to store predictions
    names = []
    all_z = []
    all_R = []
    all_full_rep = []
    all_predicted_tfs = []
    all_predicted_params = []

    #Get the names of the light curves and the transfer_function and params
    transfer_functions_test = []
    params_test = []
    names_test = []
    for i in testLoader:
        name = i['lcName'][0].split('_')[0]
        names.append(name)
        names_test.append(name)
        if tf_and_param:
            transfer_functions_test.append(i['transfer_function'][0].numpy())
            params_test.append(i['parameters'][0].numpy())
    params_test = np.array(params_test)
    transfer_functions_test = np.array(transfer_functions_test)

    #prediction and ploting of test data
    testMetrics,z_test,R_test, agg_R_z_test,predicted_params_test,predicted_tf_test,predicted_classes =por.plot_test_data(model, testLoader, criterion, mseMetric, plot_function2, device, tr,OUTPUT_PATH,beta_param = beta_param,beta_classifier=0,beta_tf=beta_tf)

    #Save all the results to arrays
    all_z.append(z_test)
    all_R.append(R_test)
    all_full_rep.append(agg_R_z_test)
    all_predicted_tfs.append(predicted_tf_test)
    all_predicted_params.append(predicted_params_test)

    #saving test metrics
    savetest=por.save_test_metrics(OUTPUT_PATH, testMetrics)

    #Get the names of the light curves and the transfer_function and params
    transfer_functions_train = []
    params_train = []
    names_train = []
    for i in trainLoader:
        name = i['lcName'][0].split('_')[0]
        names.append(name)
        names_train.append(name)
        if tf_and_param:
            transfer_functions_train.append(i['transfer_function'][0].numpy())
            params_train.append(i['parameters'][0].numpy())
    params_train = np.array(params_train)
    transfer_functions_train = np.array(transfer_functions_train)

    #prediction and ploting of train data
    trainMetrics,z_train,R_train,agg_R_z_train,predicted_params_train,predicted_tf_train,predicted_classes =por.plot_train_light_curves(model, trainLoader,criterion, mseMetric, plot_function2, device,tr,OUTPUT_PATH,beta_param = beta_param,beta_classifier=0,beta_tf=beta_tf)

    #Save all the results to arrays
    all_z.append(z_train)
    all_R.append(R_train)
    all_full_rep.append(agg_R_z_train)
    all_predicted_tfs.append(predicted_tf_train)
    all_predicted_params.append(predicted_params_train)    

    #Save the train metrics
    savetrain=por.save_train_metrics(OUTPUT_PATH, trainMetrics)

    #Get the names of the light curves and the transfer_function and params
    transfer_functions_val = []
    params_val = []
    names_val = []
    for i in valLoader:
        name = i['lcName'][0].split('_')[0]
        names.append(name)
        names_val.append(name)
        if tf_and_param:
            transfer_functions_val.append(i['transfer_function'][0].numpy())
            params_val.append(i['parameters'][0].numpy())
    params_val = np.array(params_val)
    transfer_functions_val = np.array(transfer_functions_val)

    #Prediction and plotting of val data
    valMetrics,z_val,R_val,agg_R_z_val,predicted_params_val,predicted_tf_val,predicted_classes =por.plot_val_curves(model, valLoader,criterion, mseMetric, plot_function2, device,tr,OUTPUT_PATH,beta_param = beta_param,beta_classifier=0,beta_tf=beta_tf)    

    #Save all the results to arrays
    all_z.append(z_val)
    all_R.append(R_val)
    all_full_rep.append(agg_R_z_val)
    all_predicted_tfs.append(predicted_tf_val)
    all_predicted_params.append(predicted_params_val)

    for num in [0,1,2]:
        if num == 0:
            full_rep_test = []
            for i in range(len(all_full_rep[num])):
                full_rep_test.append(np.array(all_full_rep[0][i][0])[0,:,:].mean(axis = 0))
            z_test = all_z[0][:,0,0,:]
            full_rep_test = np.array(full_rep_test)
        elif num == 1:
            full_rep_train = []
            for i in range(len(all_full_rep[num])):
                full_rep_train.append(np.array(all_full_rep[1][i][0])[0,:,:].mean(axis = 0))
            full_rep_train = np.array(full_rep_train)
            z_train = all_z[1][:,0,0,:]
        elif num == 2:
            full_rep_val = []
            for i in range(len(all_full_rep[num])):
                full_rep_val.append(np.array(all_full_rep[2][i][0])[0,:,:].mean(axis = 0))
            full_rep_val = np.array(full_rep_val)
            z_val = all_z[2][:,0,0,:]
    all_full_rep_bands.append([full_rep_test,full_rep_train,full_rep_val])
    all_z_bands.append([z_test,z_train,z_val])
    all_transfers_funcs.append([transfer_functions_test,transfer_functions_train,transfer_functions_val])
    names_all = [names_test,names_train,names_val]

In [ ]:
names = np.concatenate(names_all)

In [ ]:
full_rep_test = np.concatenate((all_full_rep_bands[0][0],all_full_rep_bands[1][0],all_full_rep_bands[2][0],all_full_rep_bands[3][0],all_full_rep_bands[4][0]),axis = 1)
full_rep_train = np.concatenate((all_full_rep_bands[0][1],all_full_rep_bands[1][1],all_full_rep_bands[2][1],all_full_rep_bands[3][1],all_full_rep_bands[4][1]),axis = 1)
full_rep_val = np.concatenate((all_full_rep_bands[0][2],all_full_rep_bands[1][2],all_full_rep_bands[2][2],all_full_rep_bands[3][2],all_full_rep_bands[4][2]),axis = 1)

z_test = np.concatenate((all_z_bands[0][0],all_z_bands[1][0],all_z_bands[2][0],all_z_bands[3][0],all_z_bands[4][0]),axis = 1)
z_train = np.concatenate((all_z_bands[0][1],all_z_bands[1][1],all_z_bands[2][1],all_z_bands[3][1],all_z_bands[4][1]),axis = 1)
z_val = np.concatenate((all_z_bands[0][2],all_z_bands[1][2],all_z_bands[2][2],all_z_bands[3][2],all_z_bands[4][2]),axis = 1)

In [ ]:
#Transfer Function
models_tf = []
for i in range(5):
    rmodel_tf,test_loss,train_curve,val_curve = train_and_evaluate(all_full_rep_bands[i],all_transfers_funcs[i],num_epochs=300,patience=50,learning_rate=1e-4,transfer_function=True,batch_size=16,scale = False,scaler_type='standard',save_path='TFs/Weekly/',num_components=3,band = 'ugriz'[i])
    models_tf.append(rmodel_tf)

In [ ]:
#Perhaps the params need to be reshaped

input_arrays = [full_rep_test.transpose(1,0,2),full_rep_train.transpose(1,0,2),full_rep_val.transpose(1,0,2)]



lr = 1e-3
num_components = 3
num_samples = 1000
scale = False
num_epochs = 3
patience = 100
batch_size = 16

model_all,test_all,train_all,val_curve,outputs_all,std_all,params_all,names_all,everything_all,weights_all,means_gauss_all,stds_gauss_all = train_and_evaluate(input_arrays,[params_test,params_train,params_val],num_epochs=num_epochs,patience=patience,learning_rate=lr,xlabels = param_names,param_names = param_columns,batch_size=batch_size,band = 'u',scale=True,scaler_type='minmax',num_components=num_components,num_samples=num_samples,transfer_function=False)